# Laboratorio corto: Gobierno de Datos aplicado al EDA del SNP

**Curso:** Big Data - UNAJ  
**Duración:** 100 minutos  
**Dataset:** Afiliados al Sistema Nacional de Pensiones - diciembre de 2025

## Propósito

Este laboratorio no reemplaza el EDA. Agrega una capa mínima de Gobierno de Datos para poder responder, antes de analizar:

1. ¿Qué datos estamos utilizando y con qué propósito?
2. ¿Qué problemas de calidad pueden afectar el análisis?
3. ¿Quién debería responder por cada problema?
4. ¿Qué evidencia debemos dejar para que el análisis sea trazable?

## Producto del equipo

Al terminar, cada equipo presentará una ficha breve con tres hallazgos, su impacto, una regla propuesta y un responsable.

## Ruta de trabajo (100 minutos)

| Bloque | Actividad | Tiempo |
|---|---|---:|
| 1 | Reconocer el activo de datos | 15 min |
| 2 | Construir metadata mínima | 15 min |
| 3 | Ejecutar reglas de calidad | 30 min |
| 4 | Interpretar y asignar responsables | 20 min |
| 5 | Privacidad, trazabilidad y presentación | 20 min |

## 0. Preparación

La carga conserva los códigos originales y utiliza tipos anulables. Esto evita confundir una ausencia con el valor cero.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

CSV_NAME = 'Afiliados_SNP_Diciembre2025.csv'
candidates = [
    Path('../../data/bronze') / CSV_NAME,
    Path('data/bronze') / CSV_NAME,
]
CSV_PATH = next((p.resolve() for p in candidates if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(f'No se encontró {CSV_NAME}')

DTYPES = {
    'id_persona': 'string',
    'sexoact': 'Int8',
    'edadact': 'Int16',
    'redadact': 'Int8',
    'estcivil': 'Int8',
    'dpto': 'Int8',
    'aportante': 'Int8',
    'tipo_afil': 'Int8',
    'tipo_dep': 'Int8',
    'nro_aportes': 'Int32',
}

df = pd.read_csv(CSV_PATH, dtype=DTYPES, low_memory=False)
print(f'Fuente: {CSV_PATH}')
print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')

## 1. Reconocer el activo de datos - 15 minutos

Gobernar empieza por declarar el contexto. Completen los valores pendientes sin buscar una definición perfecta: deben proponer una respuesta razonable y defenderla.

**Actividad 1:** editen la ficha y expliquen oralmente por qué asignaron esos roles.

In [ ]:
ficha_activo = {
    'nombre': 'Afiliados SNP - diciembre 2025',
    'fuente': 'SBS',
    'proposito_del_uso': 'EDA académico con resultados agregados',
    'data_owner': 'POR DEFINIR POR EL EQUIPO',
    'data_steward': 'POR DEFINIR POR EL EQUIPO',
    'custodian_ti': 'POR DEFINIR POR EL EQUIPO',
    'clasificacion': 'Datos personales - uso controlado',
    'restriccion': 'No publicar identificadores individuales',
}
pd.Series(ficha_activo, name='valor').to_frame()

## 2. Metadata mínima - 15 minutos

Un nombre de columna no basta para comprender el dato. Primero generamos un inventario técnico automático. Luego cada equipo documentará tres campos críticos.

In [ ]:
catalogo_tecnico = pd.DataFrame({
    'campo': df.columns,
    'tipo': [str(df[c].dtype) for c in df.columns],
    'nulos': [int(df[c].isna().sum()) for c in df.columns],
    'nulos_pct': [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    'valores_unicos': [int(df[c].nunique(dropna=True)) for c in df.columns],
})
catalogo_tecnico

In [ ]:
# ACTIVIDAD 2: completen la definición, dominio y sensibilidad de tres campos.
metadata_negocio = pd.DataFrame([
    {'campo': 'id_persona', 'definicion': 'Identificador del afiliado',
     'dominio_o_regla': 'No nulo y único', 'sensibilidad': 'Alta'},
    {'campo': 'edadact', 'definicion': 'COMPLETAR',
     'dominio_o_regla': 'COMPLETAR', 'sensibilidad': 'COMPLETAR'},
    {'campo': 'dpto', 'definicion': 'COMPLETAR',
     'dominio_o_regla': 'COMPLETAR', 'sensibilidad': 'COMPLETAR'},
])
metadata_negocio

## 3. Auditoría corta de calidad - 30 minutos

Trabajaremos con cuatro dimensiones fáciles de reutilizar en futuros notebooks:

- **Completitud:** ¿están presentes los datos necesarios?
- **Unicidad:** ¿se evita contar dos veces la misma entidad?
- **Validez:** ¿los valores respetan el formato o dominio acordado?
- **Consistencia:** ¿los campos relacionados cuentan la misma historia?

Cada regla devuelve un porcentaje de cumplimiento. Un buen porcentaje no elimina la necesidad de revisar el impacto del incumplimiento.

In [ ]:
resultados = []

def evaluar_regla(nombre, dimension, campo, cumple, impacto, responsable):
    cumple = pd.Series(cumple, index=df.index).fillna(False).astype(bool)
    total = len(cumple)
    incumplimientos = int((~cumple).sum())
    resultados.append({
        'regla': nombre,
        'dimension': dimension,
        'campo': campo,
        'cumplimiento_pct': round(cumple.mean() * 100, 2),
        'incumplimientos': incumplimientos,
        'impacto': impacto,
        'responsable_sugerido': responsable,
    })

# Completitud
evaluar_regla(
    'Identificador informado', 'Completitud', 'id_persona',
    df['id_persona'].notna() & df['id_persona'].str.strip().ne(''),
    'No se puede reconocer ni vincular al afiliado', 'Data Steward'
)

# Unicidad
evaluar_regla(
    'Identificador único', 'Unicidad', 'id_persona',
    ~df['id_persona'].duplicated(keep=False),
    'Puede producir doble conteo en indicadores', 'Data Steward / TI'
)

# Validez
evaluar_regla(
    'Sexo dentro del dominio {1, 2}', 'Validez', 'sexoact',
    df['sexoact'].isin([1, 2]),
    'Distorsiona la segmentación por sexo', 'Data Owner / Steward'
)
evaluar_regla(
    'Edad entre 14 y 110 años', 'Validez', 'edadact',
    df['edadact'].between(14, 110),
    'Distorsiona promedios y rangos de edad', 'Data Owner / Steward'
)
evaluar_regla(
    'Departamento dentro del dominio 1..25', 'Validez', 'dpto',
    df['dpto'].between(1, 25),
    'Afecta comparaciones geográficas', 'Data Steward'
)

# Consistencia entre campos relacionados
evaluar_regla(
    'Primer periodo no posterior al último', 'Consistencia',
    'primer_periodo / ultimo_periodo',
    df['primer_periodo'].notna() & df['ultimo_periodo'].notna()
    & (df['primer_periodo'] <= df['ultimo_periodo']),
    'Produce trayectorias de aporte imposibles', 'Data Steward / TI'
)

resultado_calidad = pd.DataFrame(resultados).sort_values(
    ['cumplimiento_pct', 'regla']
).reset_index(drop=True)
resultado_calidad

### Actividad 3A: crear una regla propia

Elijan uno de estos campos: `aportante`, `tipo_afil`, `tipo_dep`, `remuneracion`, `nro_aportes` o `monto_aportes`. Definan una regla, su impacto y el responsable.

La celda siguiente funciona y sirve como plantilla. Cambien sus valores antes de ejecutarla.

In [ ]:
# EJEMPLO EDITABLE: reemplacen esta regla por la propuesta del equipo.
evaluar_regla(
    'Número de aportes no negativo',
    'Validez',
    'nro_aportes',
    df['nro_aportes'].ge(0),
    'Un valor negativo vuelve incoherente el historial de aportes',
    'Data Steward / TI',
)

resultado_calidad = pd.DataFrame(resultados).drop_duplicates('regla').sort_values(
    ['cumplimiento_pct', 'regla']
).reset_index(drop=True)
resultado_calidad

## 4. Diagnóstico y responsables - 20 minutos

**Actividad 4:** seleccionen tres hallazgos. No basta con decir que existe un error: expliquen qué decisión o indicador podría resultar afectado.

Roles de referencia:

- **Data Owner:** aprueba el significado, la regla y el nivel de calidad esperado.
- **Data Steward:** monitorea calidad, documenta y coordina correcciones.
- **Custodian / TI:** implementa validaciones, almacenamiento, permisos y procesos.
- **Data User:** utiliza el dato respetando definiciones y restricciones.

In [ ]:
ficha_diagnostico = resultado_calidad.nsmallest(3, 'cumplimiento_pct')[
    ['regla', 'dimension', 'campo', 'incumplimientos',
     'cumplimiento_pct', 'impacto', 'responsable_sugerido']
].copy()

ficha_diagnostico['accion_propuesta_por_el_equipo'] = 'COMPLETAR'
ficha_diagnostico

## 5. Privacidad y trazabilidad mínimas - 10 minutos

Gobierno de Datos también significa evitar exposiciones innecesarias y dejar evidencia del origen del análisis. Para una clase no necesitamos una plataforma de catálogo: basta con aplicar dos prácticas repetibles.

In [ ]:
# Práctica 1: no mostrar el identificador completo en ejemplos o capturas.
muestra_segura = df[['id_persona', 'edadact', 'dpto', 'aportante']].head().copy()
muestra_segura['id_persona'] = (
    muestra_segura['id_persona'].str[:4] + '********' +
    muestra_segura['id_persona'].str[-3:]
)
muestra_segura

In [ ]:
# Práctica 2: registrar el origen y el momento de ejecución.
registro_trazabilidad = pd.Series({
    'archivo_fuente': CSV_PATH.name,
    'ruta_fuente': str(CSV_PATH),
    'tamano_bytes': CSV_PATH.stat().st_size,
    'filas_leidas': len(df),
    'columnas_leidas': df.shape[1],
    'fecha_ejecucion_utc': datetime.now(timezone.utc).isoformat(),
    'transformaciones': 'carga tipada; perfilado; reglas de calidad; enmascaramiento',
}, name='valor').to_frame()
registro_trazabilidad

## 6. Presentación del equipo - 10 minutos

Cada equipo dispone de **2 minutos** y una sola tabla o captura. Debe responder:

1. ¿Cuál fue el hallazgo más importante?
2. ¿Qué dimensión de calidad representa?
3. ¿Cómo podría afectar una conclusión del EDA?
4. ¿Qué regla proponen?
5. ¿Quién aprueba la regla y quién la implementa?

In [ ]:
print('FICHA FINAL DEL EQUIPO')
print('=' * 80)
print(f'Activo: {ficha_activo["nombre"]}')
print(f'Propósito: {ficha_activo["proposito_del_uso"]}')
print(f'Reglas evaluadas: {len(resultado_calidad)}')
print(f'Registros evaluados: {len(df):,}')
print('\nTres hallazgos priorizados:')
display(ficha_diagnostico)
print('\nAntes de entregar: reemplacen todos los textos COMPLETAR o POR DEFINIR.')

---
## Checklist reutilizable para futuros notebooks

Antes de publicar un EDA, verifiquen:

- [ ] Declaramos la fuente y el propósito del dato.
- [ ] Conservamos identificadores como texto y usamos tipos adecuados.
- [ ] Documentamos los campos críticos y sus dominios.
- [ ] Medimos nulos, duplicados, valores inválidos e inconsistencias.
- [ ] Explicamos el impacto de los problemas, no solo su cantidad.
- [ ] Asignamos un responsable para aprobar e implementar cada regla.
- [ ] Evitamos mostrar datos personales innecesarios.
- [ ] Registramos archivo fuente, fecha, filas y transformaciones.

Este checklist constituye el **mínimo viable de Gobierno de Datos** para los próximos trabajos del curso.